In [1]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math

In [2]:
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
        for p in group["params"]:
            if p.grad is None:
                continue
            state = self.state[p] # Get state associated with p.
            t = state.get("t", 0) # Get iteration number from the state, or initial value.
            grad = p.grad.data # Get the gradient of loss with respect to p.
            p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
            state["t"] = t + 1 # Increment iteration number.
        return loss

In [10]:
for lr in [1, 1e1, 1e2, 1e3]:
    print('-'*50)
    print(f'{lr=}')
    weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
    opt = SGD([weights], lr=lr)
    for t in range(10):
        opt.zero_grad() # Reset the gradients for all learnable parameters.
        loss = (weights**2).mean() # Compute a scalar loss value.
        print(f'{loss.cpu().item():.0f}')
        loss.backward() # Run backward pass, which computes gradients.
        opt.step() # Run optimizer step.

--------------------------------------------------
lr=1
21
21
20
20
19
19
19
18
18
18
--------------------------------------------------
lr=10.0
26
16
12
9
8
6
5
5
4
3
--------------------------------------------------
lr=100.0
23
23
4
0
0
0
0
0
0
0
--------------------------------------------------
lr=1000.0
28
10044
1734748
192972224
15630747648
986479132672
50642613698560
2178862007975936
80308200443740160
2578785775692808192
